# 🤟 Upgraded HST-GNN — Sign Language Translation

**Nâng cấp từ:** arxiv 2111.07258 (Kan et al., 2021)

## Những gì đã nâng cấp:
| Component | Bài gốc | Bản nâng cấp |
|---|---|---|
| Feature extraction | ResNet-152 + TVL1-flow | MediaPipe keypoints (offline, free) |
| Graph encoder | HST-GNN (bilinear adj) | HST-GNN Lite (cosine adj, pre-LN) |
| Temporal | 2×LSTM | 1D Conv + Transformer |
| Text decoder | 2-stage LSTM | mBART-50 pretrained |
| Loss | CTC + CE | CTC + CE (joint, ổn định hơn) |
| LR schedule | Fixed | Warmup + Cosine Decay |
| Augmentation | None | 7 loại augmentation |
| Mixed Precision | No | AMP (2× faster) |
| Dataset | PHOENIX only | Multi-dataset (Kaggle support) |


## 0. Mount Google Drive (để auto-save checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BACKUP = '/content/drive/MyDrive/sign_language_checkpoints'
import os
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f'Drive backup: {DRIVE_BACKUP}')

## 1. Check GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

## 2. Install Dependencies

In [ ]:
%%capture
# Core ML
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.40.0 accelerate

# Kaggle data access
!pip install kagglehub[pandas-datasets]

# Keypoint extraction
!pip install mediapipe opencv-python-headless

# Utilities
!pip install pyyaml tqdm einops

print('✓ All dependencies installed')

## 3. Upload Code Files

In [ ]:
# Upload các file .py từ máy local
# Hoặc clone từ GitHub nếu đã push lên:
# !git clone https://github.com/YOUR_USERNAME/sign-language-hstgnn
# %cd sign-language-hstgnn

# Manual upload:
from google.colab import files

print('Uploading Python files...')
uploaded = files.upload()  # Upload: train.py, model.py, dataset.py, config.py, utils.py, vocabulary.py, scheduler.py

import os
os.makedirs('sign_language', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'sign_language/{fname}')
    print(f'Moved: {fname}')

%cd sign_language
!ls -la

## 4. Setup Kaggle API & Download Dataset

In [ ]:
# Upload kaggle.json (lấy từ https://www.kaggle.com/settings > API > Generate New Token)
from google.colab import files

print('Upload kaggle.json...')
uploaded = files.upload()

import os, shutil
os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.config/kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.config/kaggle/kaggle.json'), 0o600)
print('✓ Kaggle API configured')

In [ ]:
# ── Option A: PHOENIX-2014-T (German Sign Language) ──────────
# Dataset yêu cầu đăng ký tại: https://www-i6.informatik.rwth-aachen.de/~koller/RWTH-PHOENIX/
# Sau khi có keypoints đã extract, upload trực tiếp

# ── Option B: Sign Language datasets từ Kaggle ───────────────
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Dataset 1: Sign Language MNIST (quick test)
print('Downloading Sign Language MNIST...')
df_mnist = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'datamunge/sign-language-mnist',
)
print(f'MNIST shape: {df_mnist.shape}')
print(df_mnist.head())

# Dataset 2: ASL Dataset
# path = kagglehub.dataset_download('grassknoted/asl-alphabet')
# print(f'ASL downloaded to: {path}')

## 5. Prepare Data

In [ ]:
import sys
sys.path.insert(0, '.')

import json, os
import numpy as np
from pathlib import Path

os.makedirs('data', exist_ok=True)

# ── Convert MNIST → our format ───────────────────────────────
# (Đây là ví dụ; thay bằng PHOENIX nếu có)

def mnist_to_samples(df, split):
    """Convert Sign Language MNIST to our JSON format."""
    samples = []
    pixel_cols = [c for c in df.columns if c.startswith('pixel')]
    
    for i, row in df.iterrows():
        label = int(row['label'])
        label_char = chr(ord('A') + label) if label < 26 else str(label)
        
        # Create fake keypoints (28x28 pixels → reshape → use as skeleton proxy)
        # In production: extract MediaPipe keypoints from image
        pixels = row[pixel_cols].values.astype(np.float32) / 255.0
        img_28x28 = pixels.reshape(28, 28)
        
        # Fake keypoints: 543 nodes, 4 dims (x, y, z, vis)
        fake_kps = np.zeros((1, 543, 4), dtype=np.float32)  # T=1 frame
        # Fill with pixel values as proxy features (for demo)
        n_fill = min(543 * 4, len(pixels))
        fake_kps[0, :n_fill//4, :] = pixels[:n_fill].reshape(-1, 4)[:n_fill//4]
        
        samples.append({
            'id': f'mnist_{split}_{i}',
            'keypoints': fake_kps.tolist(),
            'gloss': label_char,
            'text': label_char,
            'dataset': 'sign_language_mnist',
            'split': split
        })
    return samples

# Split 80/10/10
from sklearn.model_selection import train_test_split
# Nếu không có sklearn: manual split
try:
    from sklearn.model_selection import train_test_split
    train_df, temp_df = train_test_split(df_mnist, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
except ImportError:
    n = len(df_mnist)
    train_df = df_mnist.iloc[:int(n*0.8)]
    val_df = df_mnist.iloc[int(n*0.8):int(n*0.9)]
    test_df = df_mnist.iloc[int(n*0.9):]

train_samples = mnist_to_samples(train_df, 'train')
val_samples = mnist_to_samples(val_df, 'val')
test_samples = mnist_to_samples(test_df, 'test')

with open('data/train.json', 'w') as f:
    json.dump(train_samples, f)
with open('data/val.json', 'w') as f:
    json.dump(val_samples, f)
with open('data/test.json', 'w') as f:
    json.dump(test_samples, f)

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')
print('Sample:', json.dumps(train_samples[0], default=str)[:200])

In [ ]:
# ── Build Vocabularies ───────────────────────────────────────
from vocabulary import Vocabulary

gloss_vocab = Vocabulary.build_from_json('data/train.json', 'gloss')
text_vocab = Vocabulary.build_from_json('data/train.json', 'text')

gloss_vocab.save('data/gloss_vocab.json')
text_vocab.save('data/text_vocab.json')

print(f'Gloss vocab size: {len(gloss_vocab)}')
print(f'Text vocab size:  {len(text_vocab)}')

## 6. Configure Training

In [ ]:
import yaml, os

os.makedirs('configs', exist_ok=True)

# Config cho Colab T4 (12GB VRAM)
config_colab = {
    # Data
    'train_data_path': 'data/train.json',
    'val_data_path':   'data/val.json',
    'test_data_path':  'data/test.json',
    'gloss_vocab_path': 'data/gloss_vocab.json',
    'text_vocab_path':  'data/text_vocab.json',
    'num_keypoints': 543,
    'keypoint_dim': 4,
    'max_seq_len': 256,       # Giảm xuống 256 cho Colab (tiết kiệm VRAM)
    'max_text_len': 64,
    'num_workers': 2,

    # Model (Lite version cho Colab)
    'd_model': 256,
    'num_graph_layers': 2,
    'num_temporal_layers': 3,
    'num_heads': 8,
    'dropout': 0.1,
    'decoder_name': 'facebook/mbart-large-50',
    'gradient_checkpointing': True,   # Phải True trên Colab!

    # Training
    'epochs': 50,
    'batch_size': 4,              # Nhỏ để vừa VRAM
    'eval_batch_size': 8,
    'gradient_accumulation_steps': 8,  # Effective batch = 32
    'lr': 5e-4,
    'pretrained_lr_scale': 0.05,   # mBART học với LR = 2.5e-5
    'weight_decay': 1e-4,
    'max_grad_norm': 1.0,
    'warmup_ratio': 0.1,
    'patience': 10,
    'use_amp': True,              # Mixed precision BẮT BUỘC cho Colab
    'seed': 42,

    # Loss
    'lambda_ctc': 0.5,
    'lambda_ce': 0.5,

    # Augmentation
    'aug_flip_prob': 0.5,
    'aug_noise_std': 0.01,
    'aug_scale_range': [0.85, 1.15],
    'aug_rotation': 15.0,
    'aug_time_stretch': [0.8, 1.2],
    'aug_frame_drop': 0.1,
    'aug_joint_drop': 0.05,

    # Logging
    'log_interval': 20,
    'output_dir': 'outputs/',
    'drive_backup': '/content/drive/MyDrive/sign_language_checkpoints'
}

with open('configs/colab.yaml', 'w') as f:
    yaml.dump(config_colab, f)

print('✓ Config saved to configs/colab.yaml')
print(yaml.dump(config_colab))

## 7. Sanity Check — Model Forward Pass

In [ ]:
import torch
from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary

config = get_config('configs/colab.yaml')
print(f'Device: {config.device}')

gloss_vocab = Vocabulary.load('data/gloss_vocab.json')
text_vocab = Vocabulary.load('data/text_vocab.json')

model = UpgradedHSTGNN(
    num_keypoints=config.num_keypoints,
    keypoint_dim=config.keypoint_dim,
    d_model=config.d_model,
    num_graph_layers=config.num_graph_layers,
    num_heads=config.num_heads,
    num_gloss_classes=len(gloss_vocab),
    text_vocab_size=len(text_vocab),
    decoder_name=config.decoder_name,
    dropout=config.dropout,
    use_gradient_checkpointing=config.gradient_checkpointing,
).to(config.device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total/1e6:.2f}M')
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Frozen params:    {(total-trainable)/1e6:.2f}M (mBART layers)')

# Forward pass test
B, T, N, C = 2, 32, 543, 4
kps = torch.randn(B, T, N, C).to(config.device)
kp_lens = torch.tensor([32, 28], device=config.device)
gloss_t = torch.randint(1, len(gloss_vocab), (B, 5)).to(config.device)
text_t = torch.randint(1, len(text_vocab), (B, 10)).to(config.device)

with torch.cuda.amp.autocast():
    out = model(kps, kp_lens, gloss_t, text_t)

print('\n✓ Forward pass successful!')
print(f"  gloss_log_probs: {out['gloss_log_probs'].shape}")
print(f"  text_logits:     {out['text_logits'].shape}")
print(f"  encoder_hidden:  {out['encoder_hidden'].shape}")

## 8. Train!

In [ ]:
# ── Kiểm tra xem có checkpoint cũ không (auto-resume) ────────
import os
latest_ckpt = 'outputs/checkpoint_latest.pt'
drive_ckpt = '/content/drive/MyDrive/sign_language_checkpoints/checkpoint_latest.pt'

resume_path = None
if os.path.exists(latest_ckpt):
    resume_path = latest_ckpt
    print(f'Auto-resuming from: {latest_ckpt}')
elif os.path.exists(drive_ckpt):
    # Restore từ Drive
    import shutil
    os.makedirs('outputs', exist_ok=True)
    shutil.copy(drive_ckpt, latest_ckpt)
    resume_path = latest_ckpt
    print(f'Restored checkpoint from Drive!')
else:
    print('Starting fresh training...')

print(f'Resume: {resume_path}')

In [ ]:
# ── Run Training ─────────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, 'train.py',
    '--config', 'configs/colab.yaml',
    '--output_dir', 'outputs/',
    '--drive_backup', '/content/drive/MyDrive/sign_language_checkpoints'
]

if resume_path:
    cmd += ['--resume', resume_path]

print('Starting training with command:')
print(' '.join(cmd))

# Chạy trong cell (output hiện real-time)
!{' '.join(cmd)}

## 9. Evaluate Best Model

In [ ]:
!python train.py \
    --config configs/colab.yaml \
    --resume outputs/checkpoint_best.pt \
    --eval_only \
    --output_dir outputs/

## 10. Visualize Training History

In [ ]:
import json
import matplotlib.pyplot as plt

with open('outputs/history.json') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
wer    = [h['wer'] for h in history]
bleu4  = [h['bleu4'] for h in history]
loss   = [h['loss'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, loss, 'b-o', markersize=3)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, wer, 'r-o', markersize=3)
axes[1].set_title('Val WER (↓ better)', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, bleu4, 'g-o', markersize=3)
axes[2].set_title('Val BLEU-4 (↑ better)', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Upgraded HST-GNN Training Progress', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBest WER:    {min(wer):.2f}%  (epoch {epochs[wer.index(min(wer))]})')
print(f'Best BLEU-4: {max(bleu4):.2f}   (epoch {epochs[bleu4.index(max(bleu4))]})')

## 11. Tips khi Colab hết free GPU

### Option A: Kaggle Notebooks (khuyến nghị)
```
1. Vào kaggle.com/code → New Notebook
2. Settings → Accelerator → GPU P100
3. Upload files lên Kaggle Dataset → import vào notebook
4. Copy checkpoint từ Drive xuống
5. Tiếp tục train (30h GPU/tuần free)
```

### Option B: Colab Pro ($10/tháng)
- A100 GPU (80GB VRAM) — có thể dùng batch_size=32, không cần gradient_accumulation
- Không bị disconnect

### Option C: Lightning.AI Studio (free 22h GPU/tháng)
```bash
# Persistent workspace, không mất file
pip install lightning-sdk
```

### Chiến lược training chia đoạn (quan trọng!):
```
Session 1 (2-3h): Pretrain graph encoder, freeze mBART
Session 2 (2-3h): Unfreeze decoder, fine-tune
Session 3 (2-3h): End-to-end fine-tune toàn bộ
```
Mỗi session lưu checkpoint lên Drive → session sau load tiếp.

## 12. Inference Demo


In [ ]:
import torch
from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary
import numpy as np

config = get_config('configs/colab.yaml')
gloss_vocab = Vocabulary.load('data/gloss_vocab.json')
text_vocab = Vocabulary.load('data/text_vocab.json')

# Load best model
model = UpgradedHSTGNN(
    num_keypoints=config.num_keypoints,
    keypoint_dim=config.keypoint_dim,
    d_model=config.d_model,
    num_graph_layers=config.num_graph_layers,
    num_heads=config.num_heads,
    num_gloss_classes=len(gloss_vocab),
    text_vocab_size=len(text_vocab),
    decoder_name=config.decoder_name,
).to(config.device)

state = torch.load('outputs/checkpoint_best.pt', map_location=config.device)
model.load_state_dict(state['model_state_dict'])
model.eval()

# Inference với một video (MediaPipe keypoints)
def translate_video(video_path: str):
    from dataset import extract_keypoints_from_video, KeypointNormalizer
    
    # Extract keypoints
    kps = extract_keypoints_from_video(video_path, output_path=None)
    if kps is None:
        return 'Error: Cannot extract keypoints from video'
    
    # Normalize
    normalizer = KeypointNormalizer()
    kps = normalizer(kps)
    
    # Prepare tensor
    kps_t = torch.from_numpy(kps).unsqueeze(0).to(config.device)  # (1, T, 543, 4)
    lengths = torch.tensor([kps.shape[0]], device=config.device)
    
    with torch.no_grad():
        out = model(kps_t, lengths)
    
    # Decode gloss
    gloss_ids = model.decode_gloss(out['gloss_log_probs'])
    gloss_text = gloss_vocab.ids_to_text(gloss_ids[0])
    
    # Decode text
    text_ids = model.decode_text(out['encoder_hidden'], out['encoder_lengths'],
                                  text_vocab, config.max_text_len)
    text = text_vocab.ids_to_text(text_ids[0])
    
    return {'gloss': gloss_text, 'translation': text}

# Test với video upload
# from google.colab import files
# uploaded = files.upload()
# result = translate_video(list(uploaded.keys())[0])
# print(result)

print('✓ Inference pipeline ready!')
print('Gọi: translate_video("path/to/video.mp4")')